In [2]:
import os
from dotenv import load_dotenv


In [3]:
load_dotenv()

True

In [4]:
os.environ["MLFLOW_TRACKING_URI"] = os.getenv("MLFLOW_TRACKING_URI")
os.environ["MLFLOW_TRACKING_USERNAME"] = os.getenv("MLFLOW_TRACKING_USERNAME")
os.environ["MLFLOW_TRACKING_PASSWORD"] = os.getenv("MLFLOW_TRACKING_PASSWORD")

In [6]:
%pwd

'c:\\Users\\SAAD TARIQ\\github_repositories\\wine-quality\\notebooks'

In [7]:
os.chdir("../")
%pwd

'c:\\Users\\SAAD TARIQ\\github_repositories\\wine-quality'

In [18]:
from dataclasses import dataclass
from pathlib import Path
from src.wine_quality_prediction.constants import *
from src.wine_quality_prediction.utils.common import read_yaml, create_directories, save_json
import os
import pandas as pd
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from urllib.parse import urlparse
import mlflow
import mlflow.sklearn
import numpy as np
import joblib

In [9]:
@dataclass
class ModelEvaluationConfig:
    root_directory: Path
    test_data_path: Path
    model_path: Path
    metric_file_name: Path
    all_parameters: dict
    target_column: str
    mlflow_uri: str

In [11]:
class ConfigurationManager:
    def __init__(self, config_file_path=CONFIG_FILE_PATH,
                 params_file_path=PARAMS_FILE_PATH,
                 schema_file_path=SCHEMA_FILE_PATH):
        self.config_file_path = read_yaml(path_to_yaml=config_file_path)
        self.params_file_path = read_yaml(path_to_yaml=params_file_path)
        self.schema_file_path = read_yaml(path_to_yaml=schema_file_path)

        create_directories([self.config_file_path.artifacts_root])

    def get_model_evaluation_config(self) -> ModelEvaluationConfig:
        config = self.config_file_path.model_evaluation
        params = self.params_file_path.elastic_net
        schema = self.schema_file_path.target_column

        create_directories([config.root_directory])

        model_evaluation_config = ModelEvaluationConfig(
            root_directory=config.root_directory,
            test_data_path=config.test_data_path,
            model_path=config.model_path,
            metric_file_name=config.metric_file_name,
            all_parameters=params,
            target_column=schema.name,
            mlflow_uri=os.getenv("MLFLOW_TRACKING_URI")
        )

        return model_evaluation_config

In [19]:
class ModelEvaluation:
    def __init__(self, config: ModelEvaluationConfig):
        self.config = config

    def evaluation_metrics(self, actual, pred):
        rmse = np.sqrt(mean_squared_error(y_true=actual, y_pred=pred))
        mae = mean_absolute_error(y_true=actual, y_pred=pred)
        r2 = r2_score(y_true=actual, y_pred=pred)
        return rmse, mae, r2
    
    def log_into_mlflow(self):
        test_data = pd.read_csv(self.config.test_data_path)
        model = joblib.load(self.config.model_path)

        test_x = test_data.drop([self.config.target_column], axis=1)
        test_y = test_data[[self.config.target_column]]

        mlflow.set_registry_uri(self.config.mlflow_uri)
        tracking_url_type_store = urlparse(mlflow.get_tracking_uri()).scheme

        with mlflow.start_run():

            predicted_qualities = model.predict(test_x)
            (rmse, mae, r2) = self.evaluation_metrics(actual=test_y, pred=predicted_qualities)

            scores = {"rmse": rmse, "mae": mae, "r2": r2}
            save_json(path=Path(self.config.metric_file_name), data=scores)

            mlflow.log_params(self.config.all_parameters)

            mlflow.log_metric("rmse", rmse)
            mlflow.log_metric("mae", mae)
            mlflow.log_metric("r2", r2)

            if tracking_url_type_store != "file":
                mlflow.sklearn.log_model(model, "model", registered_model_name="wine_quality_model")
            else:
                mlflow.sklearn.log_model(model, "model")

In [ ]:
try:
    config = ConfigurationManager()
    model_evaluation_config = config.get_model_evaluation_config()
    model_evaluation = ModelEvaluation(config=model_evaluation_config)
    model_evaluation.log_into_mlflow()
except Exception as e:
    raise e